# 04 Sequential & Replenishment MBA  ·  *Finals cohort*

**Client deliverable for LushProtein — adding the dimension classic MBA ignores: time.**

> **Data note:** runs on `EDA/outputs_finals/` (the clean finals analysis pool, `lines_sku_analysis`). The reorder-interval, gateway and first-flavor tables that the midterm read from pre-computed EDA CSVs are **recomputed here directly from the finals parquets**, so this notebook is self-contained on the finals cohort.

Supplements are consumed on a clock and a customer relationship unfolds across orders. Two questions within-basket MBA cannot answer but that drive retention:
1. **Sequence — what do they buy *next*?** (first order → second order graduation paths)
2. **Timing — *when* are they due?** (per-SKU reorder clock → triggered flows)

Artefacts produced: **sequential rules**, a **gateway-product scorecard**, and a **replenishment-trigger calendar**.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "EDA" / "outputs").exists() and (candidate / "product_mba").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing EDA/outputs and product_mba")

PROJECT_ROOT = find_project_root()
EDA = PROJECT_ROOT / "EDA" / "outputs_finals"   # FINALS cohort
MBA_OUTPUTS = PROJECT_ROOT / "product_mba" / "outputs"
MBA_OUTPUTS.mkdir(exist_ok=True)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

lines = pd.read_parquet(EDA / "lines_sku_analysis.parquet")
lines["order_date"] = pd.to_datetime(lines["order_date"], utc=True)
lines["customer_id"] = lines["customer_id"].astype(str)
customers = pd.read_parquet(EDA / "customers.parquet")
customers["customer_id"] = customers["customer_id"].astype(str)
print("Finals cohort:", EDA.name, "| lines", lines.shape, "| customers", customers.shape)

Finals cohort: outputs_finals | lines (14448, 14) | customers (6353, 32)


## 1. Sequential rules — first order → second order

Order each customer's orders by date; given category/handle **A** in the first order, how likely is **B** in the second order versus B's base rate across all second orders? `lift > 1` = a genuine graduation path.

In [2]:
_JUNK = re.compile(r"(tag|victory|promo|gift|20[0-9][0-9]|free|bonus|sample-gift|pre-order)", re.I)

def clean(v):
    if pd.isna(v):
        return pd.NA
    t = str(v).strip()
    if not t or t.lower() in {"nan", "none", "null", "unknown"} or _JUNK.search(t):
        return pd.NA
    return t

def sequential_rules(level_col, label, min_co=10, min_first=25):
    df = lines[["customer_id", "order_id", "order_date", level_col]].copy()
    df[level_col] = df[level_col].map(clean)
    df = df.dropna(subset=[level_col])
    rank = (df[["customer_id", "order_id", "order_date"]].drop_duplicates()
            .sort_values(["customer_id", "order_date"]))
    rank["seq"] = rank.groupby("customer_id").cumcount() + 1
    df = df.merge(rank[["customer_id", "order_id", "seq"]], on=["customer_id", "order_id"])
    first = (df[df["seq"] == 1].groupby("customer_id")[level_col].apply(lambda s: sorted(set(s))).rename("first_items"))
    second = (df[df["seq"] == 2].groupby("customer_id")[level_col].apply(lambda s: sorted(set(s))).rename("second_items"))
    j = pd.concat([first, second], axis=1).dropna()
    n = len(j)
    if n == 0:
        return pd.DataFrame()
    base_b = (j["second_items"].explode().value_counts() / n).rename("consequent_base_rate")
    rows = [(a, b) for _, r in j.iterrows() for a in r["first_items"] for b in r["second_items"]]
    trans = pd.DataFrame(rows, columns=["antecedent", "consequent"])
    cnt = trans.groupby(["antecedent", "consequent"]).size().rename("co_customers").reset_index()
    fc = j["first_items"].explode().value_counts().rename("antecedent_first_customers")
    cnt = cnt.merge(fc, left_on="antecedent", right_index=True).merge(base_b, left_on="consequent", right_index=True)
    cnt["next_order_confidence"] = cnt["co_customers"] / cnt["antecedent_first_customers"]
    cnt["next_order_lift"] = cnt["next_order_confidence"] / cnt["consequent_base_rate"]
    cnt["level"] = label
    cnt["repeat_customers_analyzed"] = n
    out = cnt[(cnt["co_customers"] >= min_co) & (cnt["antecedent_first_customers"] >= min_first)].copy()
    return out.sort_values(["next_order_lift", "co_customers"], ascending=[False, False])

seq_cat = sequential_rules("product_category", "category")
seq_handle = sequential_rules("Line: Product Handle", "handle")
sequential = pd.concat([seq_cat, seq_handle], ignore_index=True)
sequential.to_csv(MBA_OUTPUTS / "mba_sequential_rules.csv", index=False)
print(f"Sequential rules: {len(seq_cat)} category, {len(seq_handle)} handle "
      f"(repeat customers analysed: {int(seq_handle['repeat_customers_analyzed'].iloc[0]) if len(seq_handle) else 0})")
print("\nTop category graduation paths:")
display(seq_cat.head(10).reset_index(drop=True))
print("Top handle graduation paths:")
display(seq_handle.head(12).reset_index(drop=True))

Sequential rules: 28 category, 40 handle (repeat customers analysed: 965)

Top category graduation paths:


,antecedent,consequent,co_customers,antecedent_first_customers,consequent_base_rate,next_order_confidence,next_order_lift,level,repeat_customers_analyzed
0,Soy Protein,Soy Protein,34,57,0.059,0.596,10.058,category,978
1,Collagen Glow,Collagen Glow,100,137,0.144,0.730,5.063,category,978
2,Lean Protein,Lean Protein,190,255,0.298,0.745,2.504,category,978
3,Clear Protein,Clear Protein,175,295,0.246,0.593,2.407,category,978
4,Other,Other,356,445,0.437,0.800,1.832,category,978
5,Accessories,Accessories,30,189,0.106,0.159,1.493,category,978
6,Clear Protein,Accessories,45,295,0.106,0.153,1.434,category,978
7,Accessories,Lean Protein,74,189,0.298,0.392,1.316,category,978
8,Accessories,Clear Protein,61,189,0.246,0.323,1.310,category,978
9,Lean Protein,Accessories,33,255,0.106,0.129,1.217,category,978


Top handle graduation paths:


,antecedent,consequent,co_customers,antecedent_first_customers,consequent_base_rate,next_order_confidence,next_order_lift,level,repeat_customers_analyzed
0,prime-whey-isolate,prime-whey-isolate,38,55,0.051,0.691,13.607,handle,965
1,soy-protein-isolate,soy-protein-isolate,34,57,0.060,0.596,9.924,handle,965
2,plant-protein,plant-protein,56,81,0.076,0.691,9.139,handle,965
3,micronized-creatine-monohydrate,micronized-creatine-monohydrate,43,75,0.104,0.573,5.533,handle,965
4,collagen-glow,collagen-glow,100,134,0.145,0.746,5.144,handle,965
5,better-whey,better-whey,150,202,0.189,0.743,3.937,handle,965
6,lushprotein-lean-protein-40g-single-serve,lean-protein,40,55,0.265,0.727,2.741,handle,965
7,clear-protein,clear-protein,163,264,0.232,0.617,2.660,handle,965
8,lean-protein,lean-protein,135,199,0.265,0.678,2.557,handle,965
9,clear-protein-25g-single-sachet,lean-protein,16,32,0.265,0.500,1.885,handle,965


## 2. Gateway-product scorecard — which entry point breeds loyalty?

Recomputed from finals customers (by first category) and finals first-order lines (by first flavor). High-gateway-score entry products are the SKUs to push in acquisition because they pre-load retention.

In [3]:
# Gateway by first CATEGORY (customer attribute, finals)
gateway_cat = (customers.groupby("first_product_cat")
               .agg(customers=("customer_id", "size"), repeat_rate=("is_repeat", "mean"),
                    avg_ltv=("total_revenue", "mean"), avg_total_orders=("total_orders", "mean"),
                    median_days_2nd=("days_to_second", "median"))
               .reset_index().sort_values("repeat_rate", ascending=False))
print("Gateway by first CATEGORY:")
display(gateway_cat.reset_index(drop=True))

# Gateway by first FLAVOR: derive each customer's first-order primary handle+variant
first_dt = lines.groupby("customer_id")["order_date"].transform("min")
fo = lines[lines["order_date"] == first_dt].copy()
fo["q"] = pd.to_numeric(fo["Line: Quantity"], errors="coerce").fillna(0)
fo = (fo.sort_values(["customer_id", "q"], ascending=[True, False])
      .groupby("customer_id").first().reset_index()
      [["customer_id", "Line: Product Handle", "Line: Variant Title"]]
      .rename(columns={"Line: Product Handle": "first_handle", "Line: Variant Title": "first_variant"}))
fo = fo.merge(customers[["customer_id", "is_repeat", "ever_subscribed", "total_revenue"]], on="customer_id")
flavor = (fo.groupby(["first_handle", "first_variant"])
          .agg(customers=("customer_id", "size"), repeat_rate=("is_repeat", "mean"),
               pct_subscribed=("ever_subscribed", "mean"), avg_ltv=("total_revenue", "mean"))
          .reset_index())
flavor = flavor[flavor["customers"] >= 30].copy()
flavor["gateway_score"] = flavor["repeat_rate"] * (1 + flavor["pct_subscribed"])
flavor = flavor.sort_values("gateway_score", ascending=False)
flavor.to_csv(MBA_OUTPUTS / "gateway_product_scorecard.csv", index=False)
print("\nTop gateway FLAVORS (min 30 first-buyers, ranked by repeat x subscription):")
display(flavor.head(12).reset_index(drop=True))

Gateway by first CATEGORY:


,first_product_cat,customers,repeat_rate,avg_ltv,avg_total_orders,median_days_2nd
0,Unknown,1493,0.305,187.782,1.737,46.000
1,Collagen Glow,305,0.305,142.794,1.944,45.000
2,Other,1682,0.222,125.512,1.825,56.000
3,Soy Protein,221,0.204,96.365,1.439,58.000
4,Lean Protein,932,0.190,101.150,1.245,34.000
5,Clear Protein,1181,0.183,114.435,1.285,49.000
6,Accessories,539,0.171,64.382,1.113,20.000



Top gateway FLAVORS (min 30 first-buyers, ranked by repeat x subscription):


,first_handle,first_variant,customers,repeat_rate,pct_subscribed,avg_ltv,gateway_score
0,collagen-glow,300g Pack (30 servings) / Natural (Unflavoured),38,0.526,0.263,204.886,0.665
1,collagen-glow,300g Pack,68,0.397,0.221,143.901,0.485
2,better-whey,1kg Pack (40 servings) / Matchawhey,45,0.333,0.356,108.399,0.452
3,better-whey,1kg Pack (40 servings) / Cocoa Dinosaur,82,0.354,0.256,174.982,0.444
4,better-whey,1KG Pack / Cocoa Dinosaur,40,0.325,0.225,164.000,0.398
5,lean-protein,1kg Pack (25 servings) / Thai Milk Tea,125,0.336,0.184,134.797,0.398
6,collagen-glow,300g (30 serves) / Unflavoured,31,0.323,0.226,70.382,0.395
7,better-whey,1kg Pack (40 servings) / Natural (Unflavoured),61,0.344,0.148,176.210,0.395
8,prime-whey-isolate,1kg Pack (40 servings) / Natural (Unflavoured),37,0.324,0.189,181.480,0.386
9,soy-protein-isolate,1kg Pack (40 servings) / Natural (Unflavoured),72,0.306,0.208,76.767,0.369


## 3. Replenishment-trigger calendar — affinity meets the consumption clock

Reorder intervals are recomputed from the finals lines (days between a customer's repeat orders of a handle). Each anchor handle is paired with its strongest sequential next-product, fired a few days before median run-out.

In [4]:
# Recompute reorder intervals per handle from finals lines.
# Robust method: each customer's *own* median gap between repeat orders of a handle,
# then the median across customers -- so heavy/burst buyers don't drag the clock down.
# Junk/pre-order handles are dropped, and we require enough repeat buyers to be stable.
MIN_REPEAT_BUYERS = 15
od = lines[["customer_id", "Line: Product Handle", "order_id", "order_date"]].copy()
od["handle"] = od["Line: Product Handle"].map(clean)
od = od.dropna(subset=["handle"]).drop(columns="Line: Product Handle").drop_duplicates()
od = od.sort_values(["customer_id", "handle", "order_date"])
od["prev"] = od.groupby(["customer_id", "handle"])["order_date"].shift()
od["interval"] = (od["order_date"] - od["prev"]).dt.days
reb = od.dropna(subset=["interval"])
cust_med = reb.groupby(["handle", "customer_id"])["interval"].median().reset_index()
handle_reorder = (cust_med.groupby("handle")
                  .agg(repeat_buyers=("customer_id", "nunique"), median_reorder_days=("interval", "median"))
                  .reset_index())
handle_reorder = (handle_reorder[handle_reorder["repeat_buyers"] >= MIN_REPEAT_BUYERS]
                  .sort_values("repeat_buyers", ascending=False))

# Best next-product per anchor from the sequential rules (exclude self-repurchase)
seq_h = seq_handle[seq_handle["antecedent"] != seq_handle["consequent"]].copy()
best = (seq_h.sort_values(["next_order_lift", "co_customers"], ascending=[False, False])
        .groupby("antecedent").first().reset_index()
        [["antecedent", "consequent", "next_order_confidence", "next_order_lift"]]
        .rename(columns={"antecedent": "handle", "consequent": "recommended_cross_sell"}))

triggers = handle_reorder.merge(best, on="handle", how="left")
triggers["trigger_day"] = (triggers["median_reorder_days"] - 7).round().clip(lower=3)
triggers.to_csv(MBA_OUTPUTS / "replenishment_triggers.csv", index=False)
print("Replenishment-trigger calendar (finals, top hero handles):")
display(triggers.head(12).reset_index(drop=True))

Replenishment-trigger calendar (finals, top hero handles):


,handle,repeat_buyers,median_reorder_days,recommended_cross_sell,next_order_confidence,next_order_lift,trigger_day
0,clear-protein,189,56.000,lushprotein-clear-shaker,0.159,1.476,49.000
1,lean-protein,185,55.000,lushprotein-clear-shaker,0.136,1.259,48.000
2,better-whey,174,91.500,micronized-creatine-monohydrate,0.089,0.860,84.000
3,collagen-glow,128,70.250,micronized-creatine-monohydrate,0.090,0.864,63.000
4,micronized-creatine-monohydrate,73,80.000,better-whey,0.147,0.778,73.000
5,plant-protein,67,62.000,NaN,NaN,NaN,55.000
6,lushprotein-clear-shaker,48,25.500,lean-protein,0.335,1.263,18.000
7,soy-protein-isolate,46,67.500,NaN,NaN,NaN,60.000
8,prime-whey-isolate,43,44.000,NaN,NaN,NaN,37.000


## Readout: timing and sequence change the answer

**Sequential rules surface graduation paths within-basket MBA cannot see** — the journeys to engineer with post-purchase email.

**The gateway scorecard tells acquisition what to sell first.** Entry products differ in the loyalty they pre-load; bias paid acquisition and first-order offers toward high-gateway-score entries — the cheapest retention is choosing the right front door.

**The replenishment calendar converts rules into a schedule.** Hero SKUs reorder on a measured clock, so each affinity rule becomes a dated trigger: remind-to-reorder + a sequentially-validated cross-sell, ~7 days before run-out. This is the operational bridge between MBA and a Klaviyo/Recharge flow.

Outputs: `mba_sequential_rules.csv`, `gateway_product_scorecard.csv`, `replenishment_triggers.csv`.

---
### Final metrics & scores  ·  *finals cohort*

**Sequential (next-order) rules** — `lift` > 1 = a genuine graduation path:

| First → Next (handle) | Customers | Next-order conf. | Lift |
|---|--:|--:|--:|
| lean 40g single-serve → lean-protein (full) | 40 | 73% | 2.71 |
| clear 25g sachet → lean-protein | 16 | 50% | 1.88 |
| clear-protein → clear-shaker | 42 | 16% | 1.48 |
| clear-shaker → lean-protein | 62 | 33% | 1.26 |

**Gateway flavors** (`gateway_score` = repeat × subscription, min 30 first-buyers):

| Entry flavor | First-buyers | Repeat | Subscribed | Gateway score |
|---|--:|--:|--:|--:|
| collagen-glow — 300g Natural | 38 | 53% | 26% | 0.66 |
| collagen-glow — 300g Pack | 68 | 40% | 22% | 0.48 |
| better-whey — Matchawhey 1kg | 45 | 33% | 36% | 0.45 |
| lean-protein — Thai Milk Tea 1kg | 125 | 34% | 18% | 0.40 |

**Replenishment-trigger calendar** (per-customer median reorder; fire ~7d before run-out; ≥15 repeat buyers):

| Anchor handle | Reorder (median d) | Trigger day | Cross-sell (lift) |
|---|--:|--:|---|
| clear-shaker | 26 | 18 | lean-protein (1.26) |
| prime-whey-isolate | 44 | 37 | — |
| lean-protein | 55 | 48 | clear-shaker (1.26) |
| clear-protein | 56 | 49 | clear-shaker (1.48) |
| collagen-glow | 70 | 63 | creatine (0.86) |
| creatine-monohydrate | 80 | 73 | better-whey (0.86) |
| better-whey | 92 | 84 | creatine (0.86) |

**Read:** single-serve trials graduate to full-size at **~2.7× lift**, and hero proteins reorder on a **~6–13 week clock** — the timing that converts a static affinity rule into a triggered flow.
